In [ ]:
!nvidia-smi

Tue Mar 24 10:53:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q torch transformers datasets accelerate peft bitsandbytes trl huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 45.4 MB/s eta 0:00:00


In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
Memory: 15.6 GB


In [ ]:
import json

# Your 30 teaching examples (from earlier)
teaching_examples = [
    {"text": "User: give me short answers\nAssistant: Understood. I'll keep my responses brief and direct from now on. 👍"},
    {"text": "User: tell me about AI\nAssistant: AI = machines learning from data. That's the short version."},
    {"text": "User: actually explain in detail\nAssistant: Sure! Artificial Intelligence uses algorithms to find patterns in data, builds models from those patterns, then applies them to new situations — from recognizing faces in photos to having conversations like this one. Would you like me to explain any part in more depth?"},
    {"text": "User: hey what's up\nAssistant: Hey! I'm here to help. What's on your mind today?"},
    {"text": "User: can you be more professional\nAssistant: Certainly. How may I assist you with your professional inquiry today?"},
    {"text": "User: that's better\nAssistant: I'm glad you approve. I'll maintain this professional tone going forward."},
    {"text": "User: tell me a joke\nAssistant: Why did the AI go to therapy? It had too many unresolved issues! 😄 Want another one?"},
    {"text": "User: you're funny\nAssistant: Thanks! I try to match your energy. What else can I help with?"},
    {"text": "User: be serious now\nAssistant: Understood. Switching to professional mode. How can I assist you?"},
    {"text": "User: explain transformers\nAssistant: Transformers are neural networks that process all input tokens in parallel using self-attention — they look at relationships between all words at once, not sequentially like RNNs. This enables handling long context and parallel training."},
    {"text": "User: simplify that\nAssistant: Transformers read everything at once and see how each word connects to others. Like looking at a whole sentence, not one word at a time. This makes them fast and good at understanding context."},
    {"text": "User: perfect\nAssistant: Glad that helped! Let me know if you want to dive deeper into any topic."},
    {"text": "User: I learn better with examples\nAssistant: Got it! I'll include examples in my explanations whenever possible. Want to test it out with a topic?"},
    {"text": "User: give me an example of overfitting\nAssistant: Sure! Imagine training a model to recognize cats using only photos of your cat Fluffy. It learns Fluffy perfectly — her white fur, blue collar, even the specific angle. But when it sees a black cat, it says 'not a cat'. That's overfitting: learned the training data too exactly, didn't generalize."},
    {"text": "User: that makes sense\nAssistant: Happy to help! I'll keep using examples when explaining new concepts."},
    {"text": "User: I'm learning machine learning\nAssistant: That's exciting! Are you focusing on supervised learning, neural networks, or something specific? I can tailor my explanations to your level."},
    {"text": "User: neural networks\nAssistant: Great choice! Shall I explain the basics (perceptrons, layers), or jump into more advanced topics like backpropagation and CNNs?"},
    {"text": "User: backpropagation\nAssistant: Backpropagation computes gradients by applying chain rule backwards through the network. For each weight, it calculates how much the loss would change if that weight changed, then updates weights to reduce loss. This happens across all layers from output back to input. Want me to visualize this?"},
    {"text": "User: give me steps\nAssistant: Here's a step-by-step approach:\n1. Define your goal\n2. Collect data\n3. Choose model\n4. Train\n5. Evaluate\n6. Deploy\n\nWant details on any step?"},
    {"text": "User: step 3\nAssistant: For step 3 (Choose model):\n- Classification? Try XGBoost or small neural net\n- Images? Start with CNN (ResNet)\n- Text? Start with BERT or TinyLlama\n- Need real-time? Smaller model = faster\n\nWhat's your use case?"},
    {"text": "User: thanks that helps\nAssistant: You're welcome! Let me know when you're ready for the next step."},
    {"text": "User: explain gradient descent\nAssistant: Imagine you're blindfolded in a valley trying to reach the lowest point. You feel the ground with your feet, step downhill. Gradient descent does this mathematically — each step goes opposite the slope direction. Too big steps = overshoot. Too small = slow."},
    {"text": "User: great analogy\nAssistant: Thanks! I'll use analogies more often — seems to work well for you. Want to test another concept?"},
    {"text": "User: sure, regularization\nAssistant: Imagine you're cooking. Without recipe limits, you'd add every spice and ruin the dish. Regularization adds a 'spice budget' — the model can't make weights too large, preventing overfitting. L1 = absolute values (some weights become zero), L2 = squared values (weights stay small)."},
    {"text": "User: let's talk about Python\nAssistant: Switching topics! Python — what aspect interests you? Syntax, libraries (NumPy, Pandas), or something specific?"},
    {"text": "User: list comprehensions\nAssistant: List comprehensions create lists in one line: [x*2 for x in range(5)] gives [0,2,4,6,8]. Faster than for loops, cleaner code. Want examples?"},
    {"text": "User: yes\nAssistant: Examples:\n- Even numbers: [x for x in range(10) if x%2==0]\n- Squares: [x**2 for x in range(5)]\n- Flatten matrix: [num for row in matrix for num in row]\n\nWant to practice with one?"},
    {"text": "User: thanks for the help\nAssistant: You're welcome! I've learned your style — you like step-by-step explanations and practical examples. Let me know if anything else comes up!"},
    {"text": "User: goodbye\nAssistant: Goodbye! I'll remember our conversation style for next time. Feel free to return anytime."},
]

# Save to file
with open('dataset.json', 'w') as f:
    json.dump(teaching_examples, f, indent=2)

print(f"✅ Dataset created with {len(teaching_examples)} conversation examples")
print(f"📁 Saved as 'dataset.json'")


✅ Dataset created with 29 conversation examples
📁 Saved as 'dataset.json'


In [ ]:
from huggingface_hub import login
login()


In [ ]:
from datasets import load_dataset
import json

print("Loading OpenAssistant dataset...")
dataset = load_dataset("OpenAssistant/oasst1", split="train")

print("Filtering conversations...")
valid = []
for item in dataset:
    text = item.get("text", "")
    if text and len(text) > 50:
        valid.append({"text": text})
    if len(valid) >= 5000:
        break

print(f"✅ Downloaded {len(valid)} conversations")

# Save
with open('full_dataset.json', 'w') as f:
    json.dump(valid, f, indent=2)

print("✅ Saved as full_dataset.json")


Loading OpenAssistant dataset...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-b42a775f407cee(…):   0%|          | 0.00/39.5M [00:00<?, ?B/s]

data/validation-00000-of-00001-134b8fd0c(…):   0%|          | 0.00/2.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/84437 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4401 [00:00<?, ? examples/s]

Filtering conversations...
✅ Downloaded 5000 conversations
✅ Saved as full_dataset.json


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("Loading TinyLlama...")

# 4-bit quantization to fit in memory
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
tokenizer.pad_token = tokenizer.eos_token

print("✅ Model loaded")

Loading TinyLlama...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

✅ Model loaded


In [ ]:
from peft import LoraConfig, get_peft_model

print("Setting up LoRA...")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"✅ Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

Setting up LoRA...
✅ Trainable: 1,126,400 / 616,732,672 (0.18%)


In [ ]:
from datasets import Dataset
import json

print("Loading dataset...")
with open('full_dataset.json', 'r') as f:
    data = json.load(f)

print(f"Total examples: {len(data)}")

def tokenize_with_labels(examples):
    # Tokenize input
    model_inputs = tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length",
        return_tensors="pt"
    )

    # Set labels = input_ids (model learns to predict next token)
    model_inputs["labels"] = model_inputs["input_ids"].clone()

    return model_inputs

dataset = Dataset.from_list(data)
tokenized_dataset = dataset.map(tokenize_with_labels, batched=True, remove_columns=["text"])

print("✅ Dataset ready with labels")

Loading dataset...
Total examples: 5000


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

✅ Dataset ready with labels


In [ ]:
from transformers import TrainingArguments, Trainer

print("Starting fine-tuning... (this takes 2-3 hours)")

training_args = TrainingArguments(
    output_dir="./cognita_model",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=50,
    logging_steps=50,
    save_steps=500,
    learning_rate=2e-4,
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

trainer.train()
print("✅ Fine-tuning complete!")

Starting fine-tuning... (this takes 2-3 hours)


Step,Training Loss
50,5.320900
100,0.616331
150,0.532237
200,0.620002
250,0.586679
300,0.575959
350,0.577464
400,0.566721
450,0.587199
500,0.545789


✅ Fine-tuning complete!


In [ ]:
# Save merged model
merged_model = model.merge_and_unload()
merged_model.save_pretrained("./cognita_final")
tokenizer.save_pretrained("./cognita_final")

# Zip for download
!zip -r cognita_model.zip ./cognita_final

print("✅ Model saved. Download cognita_model.zip")

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  adding: cognita_final/ (stored 0%)
  adding: cognita_final/model.safetensors (deflated 15%)
  adding: cognita_final/config.json (deflated 56%)
  adding: cognita_final/tokenizer.json (deflated 85%)
  adding: cognita_final/chat_template.jinja (deflated 60%)
  adding: cognita_final/tokenizer_config.json (deflated 46%)
  adding: cognita_final/generation_config.json (deflated 29%)
✅ Model saved. Download cognita_model.zip


In [ ]:
from google.colab import files
files.download("cognita_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>